In [ ]:
from src.data_collection.osm_loaders import load_pois, load_area, load_streets


df_osm = load_pois()
df_streets = load_streets()
df_area = load_area()

In [ ]:
list_template = ['make_question_area_border', 'make_question_area_direction',
                 'make_question_area_inside', 'make_question_area_outside',
                 'make_question_point_between', 'make_question_point_near_cardinal',
                 'make_question_point_near_metric', 'make_question_point_near',
                 'make_question_point_towards', 'make_question_street_along',
                 'make_question_street_cross', 'make_question_street_opposite_side']

In [ ]:
from src.template_question.type_A import *
from collections import defaultdict
from itertools import combinations
import pandas as pd
import inspect

from src.template_question import type_A
from src.template_question.registry import REGISTRY
from src.template_question.ratio import allocate


def feature_agregation(df, nb_feature, nb_question, features_cols, dropna=True, seed=42):
    ids = df["poi_id"]
    rows = []
    for combo in combinations(features_cols, nb_feature):
        df = df.dropna(subset=list(combo))
        g = df.groupby(list(combo), dropna=dropna, observed=True)
        for values, idx in g.groups.items():
            values = values if isinstance(values, tuple) else (values,)
            rows.append({
                "features": combo,
                "values": values,
                "number features": nb_feature,
                "results_features_pois_id": list(ids.loc[idx]),
                "size": len(idx),
                **dict(zip(combo, values)),
            })

    out = pd.DataFrame(rows)
    return (out.sample(n=min(nb_question, len(out)), random_state=seed))

def question_geo_semantic(df_question, df_osm, nb_q, features_cols, ratio=None, seed=42):
    if ratio==None:
        ratio = {i: 1 for i in range(1, len(features_cols) + 1)}
    balance = allocate(nb_q, ratio)
    df_out = []
    for _, question in df_question.iterrows():
        id_poi = question["results_poi_id"]
        sub_df = df_osm[df_osm['poi_id'].isin(set(id_poi))]
        for nb_f, nb_row in balance.items():
            answers_poi = feature_agregation(sub_df, nb_f, nb_row, features_cols, seed)
            res = question.to_frame().T.merge(answers_poi, how="cross")
            df_out.append(res)
    return pd.concat(df_out, ignore_index=True)


def make_question_geo(
        df_osm, df_area=None, df_streets=None, nb_q_by_temp=100, nb_q_by_feat=10, 
        list_template=list_template, ratio=None, 
        feature_cols=["outdoor_seating", "indoor_seating", "cuisine"],
        seed=42
    ):
    sources = {"df_osm": df_osm, "df_area": df_area, "df_streets": df_streets}
    df_geo_q = pd.concat(
        [REGISTRY[name](nb_q=nb_q_by_temp,
                        **{k: v for k, v in sources.items()
                           if k in inspect.signature(REGISTRY[name]).parameters})
         for name in list_template],
        ignore_index=True)
    df_geo_sem = question_geo_semantic(df_geo_q, df_osm, nb_q_by_feat, feature_cols, ratio, seed=42)
    return df_geo_sem

    

"""
il y a un ratio pour les template
un ratio pour les features 
donc par template il doit y avoir le ratio de fetures 
"""

In [ ]:
out = make_question_geo(df_osm, df_area, df_streets, 20)